# Project country mean TMREL onto the 0.25° grid

Joins `tmrel_{cid}_summaries.nc` (one value per country name) to `country_id_mask_0_25.nc` (one integer ID per cell).

In [1]:
import re
import os
import json
import numpy as np
import pandas as pd
import xarray as xr
import config
from utils.utils import require_dir, adjust_longitude, standardise_latlon
import pathlib

In [2]:
# === Path config ===
ID_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")
TMREL_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23" / "TMREL" / "summaries")
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country", "country masks")

In [4]:
# Load look up file
lookup_file = "country_location_ids.json"  # {"Armenia": 33, "Azerbaijan": 34, ...}
lookup_path = os.path.join(ID_DIR, lookup_file)
with open(lookup_path, "r", encoding="utf-8") as f:
    country_to_id = json.load(f)

# Load country id mask
mask_file = "country_id_mask_0.25.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataset(mask_path)

Combine per-country tmrel files into one array

In [5]:
# filenames are exactly "<tmrel_<cid>_summaries.nc"
country_files = {int(cid): TMREL_DIR / f"tmrel_{cid}_summaries.nc" for cid in country_to_id.values()}

# Check which files exist and which IDs are missing
missing = [cid for cid, p in country_files.items() if not p.exists()]
print("IDs in dict with no file:", missing)
country_files = {cid: p for cid, p in country_files.items() if p.exists()}

# Combine per-country files into one array with a 'country' dim
tmrelMean = xr.concat(
    [xr.open_dataset(p)["tmrelMean"] for p in country_files.values()],
    dim=pd.Index(list(country_files.keys()), name="country"),
    join="exact",  # errors if annual_temp/daily_temp grids differ between countries
    )

IDs in dict with no file: []


In [11]:
# Put dims in a known order regardless of how the file stores them
tmrel = tmrelMean.transpose("annual_temperature", "country")

cid  = tmrel["country"].values.astype(np.int64)
temp = tmrel["annual_temperature"].values

assert len(set(cid)) == len(cid), "Duplicate country IDs in tmrel"
assert np.all(np.diff(temp) > 0), "annual_temperature must be strictly increasing"
print("annual_temperature:", temp.min(), "-", temp.max(), f"({len(temp)} values, step {set(np.diff(temp).tolist())})")

# tmrel IDs vs JSON
unknown = sorted(set(cid.tolist()) - set(country_to_id.values()))
assert not unknown, f"tmrel IDs not in the JSON: {unknown}"
print("tmrel IDs missing from JSON:", unknown, "| JSON IDs missing from tmrel:",
      sorted(country_to_id.values()[i] for i in set(country_to_id.values()) - set(cid.tolist())))

# NaNs in the table itself
nan_by_country = tmrel.isnull().sum("annual_temperature")
bad = nan_by_country[nan_by_country > 0]
print("countries with NaN TMREL entries:", {country_to_id[int(c)]: int(n) for c, n in zip(bad["country"].values, bad.values)})

annual_temperature: 6 - 28 (23 values, step {1})
tmrel IDs missing from JSON: [] | JSON IDs missing from tmrel: []
countries with NaN TMREL entries: {}


In [12]:
ids = masks["country_id"].values                       # float64 with NaN
valid = np.isfinite(ids)
ids_int = np.where(valid, ids, -1).astype(np.int64)
assert np.all(ids[valid] == np.round(ids[valid])), "Non-integer IDs in mask"

mask_ids = set(np.unique(ids_int[valid]).tolist())
assert not (mask_ids - set(country_to_id.values())), "Mask has IDs not in the JSON"

lut = np.full((len(temp), max(country_to_id.values()) + 1), np.nan, dtype=np.float32)
lut[:, cid] = tmrel.values                            # (T, C) -> columns by country ID

grid = np.full((len(temp),) + ids.shape, np.nan, dtype=np.float32)
grid[:, valid] = lut[:, ids_int[valid]]

tmrel_da = xr.DataArray(
    grid,
    coords={"annual_temperature": tmrel["annual_temperature"], "lat": masks["lat"], "lon": masks["lon"]},
    dims=("annual_temperature", "lat", "lon"),
    name="TMREL",
    attrs={**tmrel.attrs,
           "grid_note": "Country covering the largest share of each 0.25 deg cell; NaN where no country.",
           "id_source": "country_location_ids.json"},
)

out_file = "tmrel_mean_0.25_map.nc"
out_path = os.path.join(TMREL_DIR, out_file)
print(f"Saving to {out_path}")
tmrel_da.to_netcdf(out_path)

Saving to /glade/work/awells/workflow/GBD23/TMREL/summaries/tmrel_mean_0.25_map.nc
